# 08 — Predicted-tag translation: the pipeline condition

Sections 8 and 9 of the paper report an **oracle** result: entity markup taken
from the corrected projection. A deployed system has no projection at inference
time, only what the recognizer predicts. This notebook measures that.

Three conditions on the same 22,059 test sentences, all translated by models
already trained:

| Condition | Source markup |
|---|---|
| baseline | none |
| oracle | from the corrected projection |
| **predicted** | from `mizo_ner_v2` |

Two complications are handled explicitly.

**Leakage.** The recognizer trained on an 80/10/10 split; the MT test set comes
from a separate 90/5/5 split over the same corpus, so most MT test sentences
were seen during NER training. We identify the unseen subset and report it
separately. That is the number to quote.

**Markup form.** The projection marks stems (`«GPE» Aizawl «/GPE»ah`); the
recognizer tags whole tokens (`Aizawlah`). Feeding a different convention than
the MT model trained on would penalize the predicted condition for formatting
rather than tag quality, so we align to stems using a lexicon built from
training-split annotations only.

**Run from the repository root.** Kernel: `Python (tka)`. About 20 minutes.

## Cell 1: Setup

In [1]:
from pathlib import Path
import json, re, sys, time
from collections import Counter
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

MT_DATA = ROOT / "data" / "processed" / "mt_v2"
BIO     = ROOT / "data" / "processed" / "bio_v2"
CORPUS  = ROOT / "data" / "processed" / "mizo_ner_annotations_dedup_aggressive.jsonl"
MODELS  = ROOT / "models"
RES     = ROOT / "results" / "mt"

need = [MT_DATA / "test_mizo_plain.txt", MT_DATA / "test_english.txt",
        MT_DATA / "split_indices.json", BIO / "mizo_ner_train.json",
        CORPUS, MODELS / "mizo_ner_v2" / "config.json",
        MODELS / "mt_marked_v2" / "config.json",
        RES / "hyps_baseline_v2.json", RES / "hyps_marked_v2.json"]
for p in need:
    print(("  ok   " if p.exists() else "  MISS ") + p.name)
    if not p.exists():
        sys.exit("Run notebooks 03-07 first")

device = "cuda" if torch.cuda.is_available() else "cpu"
OPEN, CLOSE = "\u00ab", "\u00bb"

def write_json(obj, path, **kw):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, **kw)

print(f"\nDevice: {device}")

Repo root: C:\Users\Haulai\mizo-ner
  ok   test_mizo_plain.txt
  ok   test_english.txt
  ok   split_indices.json
  ok   mizo_ner_train.json
  ok   mizo_ner_annotations_dedup_aggressive.jsonl
  ok   config.json
  ok   config.json
  ok   hyps_baseline_v2.json
  ok   hyps_marked_v2.json

Device: cuda


## Cell 2: Test data and existing translations

In [2]:
def lines(p):
    return open(p, encoding="utf-8").read().splitlines()

src_plain = lines(MT_DATA / "test_mizo_plain.txt")
refs      = lines(MT_DATA / "test_english.txt")
N = len(refs)
hyps = {k: json.load(open(RES / f"hyps_{k}_v2.json", encoding="utf-8"))
        for k in ("baseline", "marked")}
assert len(hyps["baseline"]) == len(hyps["marked"]) == N
print(f"Test sentences: {N:,}   translations loaded for baseline and oracle")

split = json.load(open(MT_DATA / "split_indices.json"))
test_idx = split["test"]
assert len(test_idx) == N
print(f"Corpus indices for the MT test split: {len(test_idx):,}")

Test sentences: 22,059   translations loaded for baseline and oracle
Corpus indices for the MT test split: 22,059


## Cell 3: Which test sentences did the recognizer already see?

`split_indices.json` holds 0-based positions into the corpus record list; the
BIO files carry 1-based `id`. Verify the mapping on text before trusting it.

In [3]:
records = []
with open(CORPUS, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print(f"Corpus records: {len(records):,}")

# verify index -> record alignment against the MT test text
mismatch = sum(1 for k, i in enumerate(test_idx[:2000])
               if records[i]["text"].strip() != src_plain[k])
print(f"index/text mismatches in first 2,000: {mismatch}")
assert mismatch == 0, "split indices do not line up with the corpus records"

ner_train_ids = {r["id"] for r in json.load(open(BIO / "mizo_ner_train.json", encoding="utf-8"))}
print(f"NER training sentences: {len(ner_train_ids):,}")

seen = np.array([(i + 1) in ner_train_ids for i in test_idx])
n_seen, n_unseen = int(seen.sum()), int((~seen).sum())
print(f"\nMT test sentences seen during NER training : {n_seen:,} ({n_seen/N*100:.1f}%)")
print(f"MT test sentences NOT seen                  : {n_unseen:,} ({n_unseen/N*100:.1f}%)")
unseen_idx = np.where(~seen)[0]

Corpus records: 441,178
index/text mismatches in first 2,000: 0
NER training sentences: 352,941

MT test sentences seen during NER training : 17,653 (80.0%)
MT test sentences NOT seen                  : 4,406 (20.0%)


## Cell 4: Predict tags on the plain test source

In [4]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

ner_dir = MODELS / "mizo_ner_v2"
ner_tok = AutoTokenizer.from_pretrained(str(ner_dir))
ner_mdl = AutoModelForTokenClassification.from_pretrained(str(ner_dir)).to(device).eval()
id2tag  = {int(k): v for k, v in ner_mdl.config.id2label.items()}
MAX_LEN = json.load(open(ROOT / "results" / "ner" / "training_v2.json"))["max_len"]
print(f"MAX_LEN {MAX_LEN}, {len(id2tag)} labels")

def predict_tags(sentences, batch=128):
    out, t0 = [], time.time()
    for b in range(0, len(sentences), batch):
        chunk = sentences[b:b + batch]
        toks = [s.split() for s in chunk]
        enc = ner_tok(toks, is_split_into_words=True, max_length=MAX_LEN,
                      padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            pred = torch.argmax(ner_mdl(**enc).logits, dim=2).cpu().numpy()
        for j, tk in enumerate(toks):
            wid, seen_w, row = enc.word_ids(batch_index=j), set(), ["O"] * len(tk)
            for pos, w in enumerate(wid):
                if w is not None and w not in seen_w:
                    seen_w.add(w); row[w] = id2tag[int(pred[j][pos])]
            out.append(row)
        if (b // batch) % 50 == 0:
            print(f"  {b:,}/{len(sentences):,}  {(time.time()-t0)/60:.1f} min", flush=True)
    print(f"  done in {(time.time()-t0)/60:.1f} min")
    return out

pred_tags = predict_tags(src_plain)
del ner_mdl; torch.cuda.empty_cache()

n_pred = sum(sum(1 for t in row if t.startswith("B-")) for row in pred_tags)
n_orac = sum(len(records[i]["entities"]) for i in test_idx)
print(f"\nentities predicted : {n_pred:,}  ({n_pred/N:.2f} per sentence)")
print(f"entities in oracle : {n_orac:,}  ({n_orac/N:.2f} per sentence)")

MAX_LEN 96, 23 labels
  0/22,059  0.0 min
  6,400/22,059  0.2 min
  12,800/22,059  0.4 min
  19,200/22,059  0.5 min
  done in 0.6 min

entities predicted : 29,719  (1.35 per sentence)
entities in oracle : 29,634  (1.34 per sentence)


## Cell 5: Stem lexicon from training data only

Built from the projection spans of sentences **not** in the MT test set, so
nothing about the test references leaks into the alignment.

In [5]:
test_set = set(test_idx)
lex = set()
for i, rec in enumerate(records):
    if i in test_set:
        continue
    for s, e, _lab in rec.get("entities", []):
        w = rec["text"][s:e].strip()
        if w:
            lex.add(w.lower())
            for part in w.split():
                lex.add(part.lower())
print(f"stem lexicon: {len(lex):,} forms (from non-test sentences only)")

# Case markers only. Earlier drafts included -na and -te; -na is a nominalizer
# that attaches to verbs, not a case marker on proper nouns, and it wrongly
# reduced Liana to Lia. The terminal -a of male names and -i of female names are
# name-forming rather than inflectional, so a bare -a is stripped only when the
# lexicon confirms the shorter form (Sanga -> Sang, but Liana stays whole).
SUF = sorted(["ah", "an", "in", "a"], key=len, reverse=True)
TRAIL = ".,;:!?\"')"

def stem_end(token):
    """Character length of the stem portion of a whole token the tagger marked."""
    core = token.rstrip(TRAIL)
    if "-" in core:
        return len(core.split("-")[0])
    if core.lower() in lex:
        return len(core)
    for s in SUF:
        if core.lower().endswith(s) and len(core) - len(s) >= 3:
            if core[:len(core) - len(s)].lower() in lex:
                return len(core) - len(s)
    return len(core)

for t in ["Aizawlah", "Liana-an", "Liana", "Mizoramin", "Cairo.", "Sanga",
          "Hmingaah", "Unknownah"]:
    print(f"  {t:<12} -> {t[:stem_end(t)]}")
assert stem_end("Liana") == 5, "Liana must not be shortened"

stem lexicon: 57,967 forms (from non-test sentences only)
  Aizawlah     -> Aizawl
  Liana-an     -> Liana
  Liana        -> Liana
  Mizoramin    -> Mizoram
  Cairo.       -> Cairo
  Sanga        -> Sanga
  Hmingaah     -> Hminga
  Unknownah    -> Unknown


## Cell 6: Build the predicted-markup source

In [6]:
def spans_from_tags(tokens, tags, align):
    """BIO over whitespace tokens -> character spans in the rebuilt sentence."""
    spans, i = [], 0
    starts = []
    pos = 0
    for t in tokens:
        starts.append(pos); pos += len(t) + 1
    while i < len(tokens):
        if tags[i].startswith("B-"):
            lab = tags[i][2:]
            j = i + 1
            while j < len(tokens) and tags[j] == f"I-{lab}":
                j += 1
            s = starts[i]
            last = tokens[j - 1]
            e = starts[j - 1] + (stem_end(last) if align else len(last.rstrip(TRAIL)))
            if e > s:
                spans.append((s, e, lab))
            i = j
        else:
            i += 1
    return spans

def inject(text, spans):
    out = text
    for s, e, lab in sorted(spans, key=lambda x: -x[0]):
        out = out[:s] + f"{OPEN}{lab}{CLOSE} {out[s:e]} {OPEN}/{lab}{CLOSE}" + out[e:]
    return out

src_pred_aligned, src_pred_raw = [], []
for sent, tags in zip(src_plain, pred_tags):
    toks = sent.split()
    base = " ".join(toks)
    src_pred_aligned.append(inject(base, spans_from_tags(toks, tags, True)))
    src_pred_raw.append(inject(base, spans_from_tags(toks, tags, False)))

print("plain    :", src_plain[0])
print("aligned  :", src_pred_aligned[0])
print("raw      :", src_pred_raw[0])
diff = sum(1 for a, b in zip(src_pred_aligned, src_pred_raw) if a != b)
print(f"\nsentences where stem alignment changed the markup: {diff:,} ({diff/N*100:.1f}%)")

plain    : Tu nge a nih Kunga an a zawt.
aligned  : Tu nge a nih «PERSON» Kunga «/PERSON» an a zawt.
raw      : Tu nge a nih «PERSON» Kunga «/PERSON» an a zawt.

sentences where stem alignment changed the markup: 4,122 (18.7%)


## Cell 7: Agreement between predicted and oracle markup

In [7]:
oracle_spans = [{(s, e, lab) for s, e, lab in records[i]["entities"]} for i in test_idx]
pred_spans   = [set(spans_from_tags(s.split(), t, True))
                for s, t in zip(src_plain, pred_tags)]

tp = sum(len(p & o) for p, o in zip(pred_spans, oracle_spans))
fp = sum(len(p - o) for p, o in zip(pred_spans, oracle_spans))
fn = sum(len(o - p) for p, o in zip(pred_spans, oracle_spans))
prec = tp / (tp + fp) if tp + fp else 0
rec  = tp / (tp + fn) if tp + fn else 0
f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0
print(f"predicted vs oracle markup (exact span+label)")
print(f"  precision {prec:.4f}   recall {rec:.4f}   F1 {f1:.4f}")
print(f"  matched {tp:,}   spurious {fp:,}   missed {fn:,}")

for name, mask in (("seen in NER training", seen), ("unseen", ~seen)):
    idxs = np.where(mask)[0]
    t = sum(len(pred_spans[i] & oracle_spans[i]) for i in idxs)
    f_p = sum(len(pred_spans[i] - oracle_spans[i]) for i in idxs)
    f_n = sum(len(oracle_spans[i] - pred_spans[i]) for i in idxs)
    p_ = t/(t+f_p) if t+f_p else 0; r_ = t/(t+f_n) if t+f_n else 0
    print(f"  {name:<22} F1 {2*p_*r_/(p_+r_) if p_+r_ else 0:.4f}  ({len(idxs):,} sentences)")

predicted vs oracle markup (exact span+label)
  precision 0.8648   recall 0.8717   F1 0.8682
  matched 25,701   spurious 4,018   missed 3,784
  seen in NER training   F1 0.8749  (17,653 sentences)
  unseen                 F1 0.8414  (4,406 sentences)


## Cell 8: Translate the predicted-markup source

In [8]:
from transformers import AutoModelForSeq2SeqLM

mt_dir = MODELS / "mt_marked_v2"
mt_tok = AutoTokenizer.from_pretrained(str(mt_dir))
mt_mdl = AutoModelForSeq2SeqLM.from_pretrained(str(mt_dir)).to(device).eval()
BEAM, GB, MAXS, MAXG = 4, 64, 128, 128

def translate(sources):
    order = sorted(range(len(sources)), key=lambda i: len(sources[i]))
    out = [None] * len(sources); t0 = time.time()
    for b in range(0, len(order), GB):
        ids = order[b:b + GB]
        enc = mt_tok([sources[i] for i in ids], return_tensors="pt",
                     padding=True, truncation=True, max_length=MAXS).to(device)
        with torch.no_grad():
            gen = mt_mdl.generate(**enc, num_beams=BEAM, max_length=MAXG)
        for i, g in zip(ids, mt_tok.batch_decode(gen, skip_special_tokens=True)):
            out[i] = g.strip()
        if (b // GB) % 100 == 0:
            print(f"  {b:,}/{len(sources):,}  {(time.time()-t0)/60:.1f} min", flush=True)
    print(f"  done in {(time.time()-t0)/60:.1f} min")
    return out

print("--- predicted (stem-aligned) ---")
hyps["predicted"] = translate(src_pred_aligned)
print("--- predicted (raw whole-token) ---")
hyps["predicted_raw"] = translate(src_pred_raw)

del mt_mdl; torch.cuda.empty_cache()
for k in ("predicted", "predicted_raw"):
    write_json(hyps[k], RES / f"hyps_{k}_v2.json")
print("\npredicted:", hyps["predicted"][0])

--- predicted (stem-aligned) ---
  0/22,059  0.0 min
  6,400/22,059  0.7 min
  12,800/22,059  1.7 min
  19,200/22,059  3.1 min
  done in 4.2 min
--- predicted (raw whole-token) ---
  0/22,059  0.0 min
  6,400/22,059  0.7 min
  12,800/22,059  1.7 min
  19,200/22,059  3.1 min
  done in 4.2 min

predicted: Kunga asked who it was.


## Cell 9: Score all conditions

In [9]:
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF
bleu_m, chrf_m = BLEU(), CHRF()

CONDS = ["baseline", "marked", "predicted", "predicted_raw"]
LABEL = {"baseline": "no markup", "marked": "oracle (projection)",
         "predicted": "predicted, stem-aligned", "predicted_raw": "predicted, whole-token"}

def score(subset=None):
    ids = range(N) if subset is None else subset
    r = [refs[i] for i in ids]
    return {c: (bleu_m.corpus_score([hyps[c][i] for i in ids], [r]).score,
                chrf_m.corpus_score([hyps[c][i] for i in ids], [r]).score)
            for c in CONDS}

full   = score()
unseen = score(unseen_idx)

for title, sc, n in (("ALL test sentences", full, N),
                     ("NER-UNSEEN subset",  unseen, len(unseen_idx))):
    print(f"\n=== {title}  (n = {n:,}) ===")
    print(f"{'Condition':<26}{'BLEU':>8}{'ChrF':>8}{'dBLEU':>9}{'dChrF':>9}")
    print("-" * 60)
    b0, c0 = sc["baseline"]
    for c in CONDS:
        b, ch = sc[c]
        d1 = "" if c == "baseline" else f"{b-b0:+8.2f}"
        d2 = "" if c == "baseline" else f"{ch-c0:+9.2f}"
        print(f"{LABEL[c]:<26}{b:>8.2f}{ch:>8.2f}{d1:>9}{d2:>9}")


=== ALL test sentences  (n = 22,059) ===
Condition                     BLEU    ChrF    dBLEU    dChrF
------------------------------------------------------------
no markup                    49.21   68.39                  
oracle (projection)          49.53   68.57    +0.33    +0.18
predicted, stem-aligned      49.12   68.37    -0.09    -0.02
predicted, whole-token       48.23   67.99    -0.98    -0.40

=== NER-UNSEEN subset  (n = 4,406) ===
Condition                     BLEU    ChrF    dBLEU    dChrF
------------------------------------------------------------
no markup                    49.65   68.55                  
oracle (projection)          50.15   68.85    +0.51    +0.30
predicted, stem-aligned      49.61   68.54    -0.04    -0.01
predicted, whole-token       48.84   68.25    -0.81    -0.30


## Cell 10: Bootstrap, predicted versus baseline

Uses per-sentence BLEU statistics rather than re-scoring the corpus each
replicate: identical result, seconds instead of an hour.

In [10]:
def boot(cond_a, cond_b, subset=None, B=1000, seed=42):
    ids = list(range(N)) if subset is None else list(subset)
    r = [refs[i] for i in ids]
    sa = np.array(bleu_m._extract_corpus_statistics([hyps[cond_a][i] for i in ids], [r]), float)
    sb = np.array(bleu_m._extract_corpus_statistics([hyps[cond_b][i] for i in ids], [r]), float)
    # full-sample check against corpus_score
    chk = bleu_m._compute_score_from_stats(list(sa.sum(axis=0))).score
    ref = bleu_m.corpus_score([hyps[cond_a][i] for i in ids], [r]).score
    assert abs(chk - ref) < 1e-6, (chk, ref)
    rng = np.random.default_rng(seed); n = len(ids); d = np.empty(B)
    for k in range(B):
        j = rng.integers(0, n, n)
        d[k] = (bleu_m._compute_score_from_stats(list(sa[j].sum(axis=0))).score
                - bleu_m._compute_score_from_stats(list(sb[j].sum(axis=0))).score)
    lo, hi = np.percentile(d, [2.5, 97.5])
    losses = int((d <= 0).sum())
    return {"mean": float(d.mean()), "lo": float(lo), "hi": float(hi),
            "wins": B - losses, "losses": losses, "p": losses / B, "n": n}

tests = [("marked", "baseline", None, "oracle vs baseline, all"),
         ("predicted", "baseline", None, "predicted vs baseline, all"),
         ("predicted", "baseline", unseen_idx, "predicted vs baseline, unseen"),
         ("marked", "predicted", None, "oracle vs predicted, all")]

boots = {}
print(f"{'Comparison':<34}{'mean':>8}{'95% CI':>20}{'p':>9}")
print("-" * 72)
for a, b, sub, name in tests:
    res = boot(a, b, sub); boots[name] = res
    p = "< 0.001" if res["losses"] == 0 else f"{res['p']:.3f}"
    ci = f"[{res['lo']:+.3f}, {res['hi']:+.3f}]"
    print(f"{name:<34}{res['mean']:>+8.3f}{ci:>20}{p:>9}")

Comparison                            mean              95% CI        p
------------------------------------------------------------------------
oracle vs baseline, all             +0.327    [+0.139, +0.527]  < 0.001
predicted vs baseline, all          -0.088    [-0.298, +0.122]    0.805
predicted vs baseline, unseen       -0.046    [-0.521, +0.400]    0.569
oracle vs predicted, all            +0.415    [+0.334, +0.501]  < 0.001


## Cell 11: Save and emit LaTeX

In [11]:
out = {
    "test_sentences": N,
    "ner_seen": int(seen.sum()), "ner_unseen": int((~seen).sum()),
    "markup_agreement": {"precision": round(prec, 4), "recall": round(rec, 4),
                         "f1": round(f1, 4), "matched": tp, "spurious": fp, "missed": fn},
    "entities_predicted": int(n_pred), "entities_oracle": int(n_orac),
    "scores_all":    {c: {"bleu": round(full[c][0], 2), "chrf": round(full[c][1], 2)} for c in CONDS},
    "scores_unseen": {c: {"bleu": round(unseen[c][0], 2), "chrf": round(unseen[c][1], 2)} for c in CONDS},
    "bootstrap": {k: {kk: (round(vv, 4) if isinstance(vv, float) else vv)
                      for kk, vv in v.items()} for k, v in boots.items()},
}
write_json(out, RES / "predicted_tag_v2.json", indent=2)
print(f"-> {(RES/'predicted_tag_v2.json').relative_to(ROOT)}\n")

print("% ---- Table: pipeline conditions ----")
b0 = full["baseline"][0]; c0 = full["baseline"][1]
u0 = unseen["baseline"][0]
for c in CONDS:
    b, ch = full[c]; ub, _ = unseen[c]
    d  = "---" if c == "baseline" else f"${b-b0:+.2f}$"
    du = "---" if c == "baseline" else f"${ub-u0:+.2f}$"
    print(f"{LABEL[c]:<26}& {b:.2f} & {ch:.2f} & {d} & {ub:.2f} & {du} \\\\")

-> results\mt\predicted_tag_v2.json

% ---- Table: pipeline conditions ----
no markup                 & 49.21 & 68.39 & --- & 49.65 & --- \\
oracle (projection)       & 49.53 & 68.57 & $+0.33$ & 50.15 & $+0.51$ \\
predicted, stem-aligned   & 49.12 & 68.37 & $-0.09$ & 49.61 & $-0.04$ \\
predicted, whole-token    & 48.23 & 67.99 & $-0.98$ & 48.84 & $-0.81$ \\
